In [ ]:
import subprocess
import os

print("=" * 60)
print("GPU STATUS")
print("=" * 60)
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'],
    capture_output=True, text=True
)
if gpu_result.returncode == 0:
    print(gpu_result.stdout)
else:
    print("WARNING: No GPU detected. Training will use CPU only.")

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from src.cd2d_streamfunc import main as generate_data
from src.create_splits import main as create_splits
from src.plot_solutions import main as plot_solutions
from src.train_solution_autoencoder import main as solution_autoencoder
from src.train_streamfunction_autoencoder import main as streamfunction_autoencoder
from src.align_latent_spaces import main as align_latents
from src.analyze_latent_alignment import main as analyze_alignment
from src.finetune_encoder_to_latent import main as finetune_enc_to_lat
from src.finetune_decoder_from_latent import main as finetune_dec_from_lat
from src.plot_modalities import main as plot_modalities
from src.compute_errors import main as errors

In [ ]:
import ipywidgets as widgets
from IPython.display import display, display_html, Markdown
import shutil

# 1. Create random solutions: `generate_data`


This script generates datasets of steady two-dimensional convection–diffusion solutions on the square domain ($[-1,1]^2$). It uses a finite-volume discretization with constant diffusion and divergence-free advection derived from a polynomial streamfunction. Random coefficients are sampled, velocities are rescaled to a target RMS magnitude, Dirichlet boundary conditions are applied, and solutions are computed on multiple grid resolutions.

<details>
    <summary>Options</summary>

#### Options
`--n-sol` INT
Number of solution samples to generate (default: 1000).

`--levels` INT ...
Grid resolutions (n) for (n \times n) solves (default: 16 32 64 128 256).

`--workers` INT
Number of parallel worker processes (default: min(24, cpu_count)).

`--seed` INT
Random seed for reproducibility (default: 12345).

##### Streamfunction / velocity

`--n-sf` INT
Maximum total polynomial degree of the streamfunction (default: 4).

`--coeff-mode` {normal,uniform}
Distribution used for streamfunction coefficients (default: uniform).

`--vel-scale` FLOAT
Target RMS velocity magnitude after rescaling (default: 100000.0).

##### Solver / diagnostics

`--solver` {auto,scipy,pardiso}
Sparse linear solver backend (default: auto).

`--jacobi-scale`
Apply Jacobi row scaling before solving.

`--debug`
Enable debug output.

`--debug-k` INT
Index of sample to debug (default: 0).

##### Post-processing

`--center`
Mean-center each solution field.

`--l2norm`
L2-normalize each solution field.

</details>

<details>
<summary>Detailed description of script</summary>

This script solves the **2D Convection-Diffusion (CD2D)** equation on a square domain $[-1, 1]^2$. Specifically, it generates a dataset of steady-state solutions where the fluid flow (advection) is "divergence-free"—meaning the fluid is incompressible, like water.

Here is a breakdown of how the code works and the math behind it:

### 1. The Physical Model
The code solves for a scalar field $u$ (like temperature or concentration) in the steady-state equation:
    $$\nabla \cdot (\mathbf{v}u) - \nabla \cdot (D \nabla u) = 0$$
    *   **Diffusion ($D$):** Kept constant at 1.0.
    *   **Advection ($\mathbf{v}$):** The velocity field is derived from a **Streamfunction** $\psi$.

### 2. Divergence-Free Flow via Streamfunctions
To ensure the flow $\mathbf{v} = (v_x, v_y)$ is divergence-free ($\nabla \cdot \mathbf{v} = 0$), the code defines a polynomial streamfunction $\psi(x, y)$. The velocity components are then calculated as:
$$v_x = \frac{\partial \psi}{\partial y}, \quad v_y = -\frac{\partial \psi}{\partial x}$$
The script uses a polynomial $\psi$ of degree `n_sf`. For example, if $n_{sf}=2$:
    $$\psi(x, y) = a_{10}x + a_{01}y + a_{20}x^2 + a_{11}xy + a_{02}y^2$$
    The coefficients $a_{ij}$ are sampled randomly, creating unique, complex swirling flow patterns for every sample.

### 3. Numerical Scheme: Finite Volumes
*   **Grid:** It uses a **cell-centered grid**. If you specify `--levels 16 32`, it solves the problem on a $16\times16$ grid, then a $32\times32$ grid, and so on.
*   **Advection (Upwinding):** To keep the simulation stable at high speeds (the `--vel-scale` defaults to $10^5$, which is very high), it uses a **Godunov (Upwind)** scheme. This means the fluid "carries" the value of $u$ from the cell it is coming from.
*   **Boundary Conditions (BCs):** It applies **Dirichlet BCs** on all four sides. Specifically, `gaussian_all_boundaries` creates "hot spots" (Gaussian bumps) in the middle of each wall, while the corners remain "cold" (zero).

### 4. Code Structure & Efficiency
*   **`eval_velocity_from_streamfunc_coeffs`**: This is the engine that calculates the velocity. It precomputes powers of $X$ and $Y$ (e.g., $X^0, X^1, X^2...$) to quickly evaluate the polynomial derivatives at both cell centers and cell faces.
*   **Parallelization**: It uses `ProcessPoolExecutor` to solve multiple samples simultaneously across your CPU cores.
*   **Memory Mapping**: Solutions are saved using `numpy.lib.format.open_memmap`. This allows the script to write huge datasets directly to disk without crashing your RAM.
*   **Solver**: It builds a sparse matrix ($A \mathbf{u} = \mathbf{b}$) and solves it using `scipy.sparse.linalg`.

### 5. Summary of Workflow
1.  **Sample**: Generate random coefficients for the streamfunction $\psi$.
2.  **Scale**: Calculate velocity and scale it so the Root Mean Square (RMS) speed matches `--vel-scale`.
3.  **Assemble**: Build the convection-diffusion matrix using the calculated velocities.
4.  **Solve**: Find the steady-state field $u$ that satisfies the BCs.
5.  **Post-process**: Optionally center the data (subtract mean) or normalize it (L2 norm) and save to `.npy` files.

### Key Snippet Explanation: Velocity Calculation
The core logic for the physics is here:
```python
// ... existing code ...
for a, (i, j) in zip(coeffs, idx):
    if j >= 1:
        jfac = j * a
        vx_c += jfac * XcP[i]  * YcP[j - 1]
        vx_f += jfac * XefP[i] * YefP[j - 1]
    if i >= 1:
        ifac = i * a
        vy_c -= ifac * XcP[i - 1]  * YcP[j]
        vy_f -= ifac * XnfP[i - 1] * YnfP[j]
// ... existing code ...
```

This loop performs the power-rule differentiation $\frac{d}{dx} x^n = n x^{n-1}$ for every term in the polynomial to get the velocity components at every point on the grid.

</details>

In [ ]:
config_data_creation = {
    'n_sol': 200,
    'levels': [16, 32, 64, 128, 256],
    'workers': min(24, os.cpu_count()),
    'seed': 12345,
    'n_sf': 4,
    'coeff_mode': 'uniform',
    'vel_scale': 100000.0,
    'solver': 'auto',
    'jacobi_scale': False,
    'debug': False,
    'debug_k': 0,
    'center': False,
    'l2norm': False,
}

In [ ]:
generate_data(**config_data_creation)

# 2. Split into training and validation sets: `splits`

This script creates reproducible train/validation/test splits for multi-modal datasets stored as `.npy` files. It verifies that all modalities share the same number of samples, optionally pre-subsamples a fixed number of indices, shuffles deterministically, and writes consistent index splits plus metadata to a single `.npz` file.

<details>
<summary>Options</summary>

- `--data_dir PATH`
Directory containing `.npy` modality files (default: `data`).

- `--splits_file PATH`
Output splits file path (default: `<data_dir>/05_splits.npz`).

- `--val_frac FLOAT`
Fraction of samples used for validation (default: `0.20`).

- `--test_frac FLOAT`
Fraction of samples used for testing (default: `0.20`).

- `--seed INT`
Random seed for shuffling and subsampling (default: `123`).

- `--no-overwrite`
Do not overwrite an existing splits file.

- `--check`
Only load an existing splits file and print split sizes and metadata.

**Pre-subsampling**
- `--limit INT`
Use only this many samples before splitting (default: use all).
- `--subset-strategy {random,first}`
How the limited subset is chosen (default: `random`).
</details>

<details>
    <summary>Detailed description of script</summary>

This Python script is a utility for creating reproducible and consistent **train, validation, and test splits** for datasets where multiple "modalities" (different data types like images, audio, or labels) are stored in separate `.npy` (NumPy) files.

Here is a breakdown of what the code does and how it works:

### 1. The Core Purpose
In multi-modal machine learning, you often have several files (e.g., `features.npy`, `labels.npy`) that must stay synchronized. If you shuffle `features.npy` but not `labels.npy`, your data becomes useless. This script solves that by generating **indices** (index numbers) for the splits once and saving them to a central `.npz` file. All subsequent processing can then use these same indices to load the correct samples.

### 2. Key Features
*   **Consistency Check:** Before splitting, it verifies that every `.npy` file in the directory has the exact same number of samples ($N$).
*   **Deterministic Shuffling:** By using a `--seed`, it ensures that if you run the script twice, you get the exact same splits.
*   **Subsampling (Limiting):** If you have a massive dataset but only want to experiment with, say, 1,000 samples, you can use `--limit 1000`. It will pick those 1,000 samples first (either randomly or the "first" $N$) and *then* split them into train/val/test.
*   **Memory Efficiency:** It uses `mmap_mode="r"` (memory mapping) when checking files. This allows it to read the metadata/shape of huge files without actually loading the data into RAM.

### 3. Functional Breakdown
*   **`find_npy_files(data_dir)`**: Scans the folder for NumPy files.
    *   **`load_and_get_N(path)`**: Opens a file to see how many samples it contains.
*   **`compute_splits_from_index_array(...)`**: This is where the math happens. It takes the list of available indices, shuffles them, and calculates the "cut-off" points based on the percentages (`val_frac` and `test_frac`) you provide.
*   **`main()`**: The conductor. It handles the logic of checking for existing files, applying the subsampling limit, calling the split function, and saving the results.

### 4. How to use it
The script is designed to be run from the command line.

**Standard split (80% Train, 10% Val, 10% Test):**
```shell script
python create_splits.py --data_dir ./my_data --val_frac 0.1 --test_frac 0.1
```


**Small-scale experiment (Use only 500 random samples total):**
```shell script
python create_splits.py --limit 500 --subset-strategy random --seed 42
```


**Check what's inside an existing split file:**
```shell script
python create_splits.py --check
```


### 5. The Output
It saves a file (usually `05_splits.npz`) containing:
1.  `train`: An array of indices for training.
2.  `val`: An array of indices for validation.
3.  `test`: An array of indices for testing.
4.  `meta`: A dictionary containing the settings used (the seed, the original file list, etc.), which is great for experiment tracking!

The script prevents one of the most common bugs in ML: accidentally training on your test data!
</details>

In [ ]:
config_splits = {
    'data_dir': 'data',
    'splits_file': 'data/05_splits.npz',
    'val_frac': 0.2,
    'test_frac': 0.2,
    'seed': 123,
    'overwrite': True,
    'check': False,
    'limit': None,
    'subset_strategy': 'random'
}

In [ ]:
create_splits(**config_splits)

# 3. Plot solutions: `plot_solutions`

This script visualizes solution fields of a 2D PDE at multiple spatial resolutions side-by-side for qualitative comparison. It loads precomputed `.npy` arrays at different grid sizes, selects sample indices (randomly or explicitly), and either displays each sample interactively or saves one PNG image per sample with consistent layout and color scaling.

<details>
<summary>Options</summary>

- `save` (`True` / `False`)
If `True`, save figures as PNG files; otherwise show them interactively.

- `results_dir STR`
Output directory for saved figures (default: `results`).

- `selection` (`'rnd'`, list, or `None`)
`'rnd'`: random order of all samples
list: explicit sample indices to plot
`None`: plot all samples in order

- `cmap STR`
Matplotlib colormap used for images (default: `"viridis"`).

- `extent LIST`
Coordinate extent passed to `imshow` (default: `[-1, 1, -1, 1]`).

- `dpi INT`
Resolution of saved PNG files (default: `150`).

</details>

In [ ]:
config_plot_solutions = {
    'data_prefix': 'data/05_',
    'levels': [16, 32, 64, 128, 256],
    'save': False,
    'results_dir': 'results',
    'selection': [1, 2, 3],
    'cmap': 'viridis',
    'extent': [-1, 1, -1, 1],
    'dpi': 150
}

In [ ]:
plot_solutions(**config_plot_solutions)

# 4. Train an autoencoder for the 2D solution field

Trains a multi-scale Conv3D autoencoder for 2D solution fields at a fixed resolution (`u16`–`u256`). It builds a pyramid-based encoder with spatial-only downsampling, learns a compact latent representation using relative energy error (REE) loss, saves the best encoder/decoder models, and exports latent vectors for all samples in a streamed, index-aligned format.

<details>

<summary>Options</summary>

**Core**
- `--modality {u16,u32,u64,u128,u256}`
Resolution of the input data.

- `--data_file PATH`
`.npy` file with shape `(N,H,W)` or `(N,H,W,1)`.

- `--splits PATH`
Train/val/test split file (`.npz` or `.json`, default: `data/05_splits.npz`).

- `--latent INT`
Latent dimension size (default: `32`).

- `--to-level STR`
Final spatial level before flattening (e.g. `1x1`, `2x2`, `4x4`; default: `1x1`).

**Training**
- `--batch INT`
Training batch size (default: `16`).
- `--epochs INT`
Maximum training epochs (default: `10000`).
- `--lr FLOAT`
Learning rate (default: `3e-4`).
- `--patience INT`
Early stopping patience on validation loss (default: `50`).

**Architecture**
- `--c0 INT`
Initial channel count at the largest spatial scale (default: `2`).
- `--cmax INT`
Maximum channel cap (default: `128`).
- `--ch-mult INT`
Channel multiplier per downsampling stage (default: `4`).
- `--depth-mixer`
Enable additional `(3,1,1)` Conv3D mixing in residual blocks.

**Export**
- `--pred-batch INT`
Batch size for latent export (default: `8`).
- `--predict-device {gpu,cpu}`
Device used when exporting latents (default: `gpu`).


</details>

<details>

<summary>Detailed description of script</summary>

## What this code is doing (high-level)

This script is building the **encoder half** of a convolutional autoencoder that:

1. Takes a **single 2D grayscale image** of shape `(H, W, 1)` where `H=W` (e.g., 256×256).
2. Builds a **9-level multi-scale pyramid** of that image (coarse-to-fine representations).
3. Treats those 9 pyramid levels as a **“depth” dimension** to create a 3D tensor and processes it with **Conv3D** blocks.
4. Downsamples **only spatially** (H/W), not across the pyramid depth.
5. Collapses spatial dimensions via **global average over H/W** and maps the result to a **latent vector** via a Dense layer.
6. Separately defines a **stand-alone decoder** that reconstructs an image from the latent vector using **2D upsampling + Conv2D** blocks.

It also defines a custom **relative error energy (REE)** reconstruction loss.


## Loss: `ree_loss`

```python
def ree_loss(y_true, y_pred, eps=1e-8):
    num = tf.reduce_sum(tf.square(y_pred - y_true), axis=[1,2,3])
    den = tf.reduce_sum(tf.square(y_true),        axis=[1,2,3]) + eps
    return tf.reduce_mean(num / den)
```


This is a **relative squared reconstruction error**:

- For each sample:
- `num` = total squared reconstruction error
- `den` = total squared energy of the true signal
- Returns `mean(num/den)` across the batch.

Why this is useful:
- It normalizes the loss by the “size” of the target, so bright/large-energy samples don’t dominate purely because they have larger absolute values.

---

## Encoder: `build_encoder(...)`

### Input and pyramid

```python
inp = keras.Input(shape=(input_side, input_side, 1))
x = PyramidToDepth(n_levels=9)(inp)  # (B,9,H,W,1)
```


Now it’s “3D data” with:
    - depth = 9 (pyramid levels)
- height/width = input size
- channels = 1

### Stem + staged downsampling

- Start with small number of channels `c0` (default 2), cap at `cmax` (default 128).
- Then loop:
- Apply `stage3d` which downsamples H/W by 2
- Multiply channels by `ch_mult` (default 4) each stage, capped by `cmax`
- Stop when spatial side equals `to_level`

It also validates that `to_level` is reachable by repeatedly dividing by 2. Example: `256 -> 128 -> 64 -> ... -> 4 -> 2 -> 1`. If you ask for `to_level=6`, it errors.

### Collapse to latent vector

Once spatial size is small enough (`to_level×to_level`):

1. Spatial average: `(B, D, to_level, to_level, ch) -> (B, D, ch)`
2. Flatten: `(B, D, ch) -> (B, D*ch)`
3. Dense: `(B, D*ch) -> (B, latent_dim)`

Result: `encoder(image) -> z` latent vector.

---

## Decoder: `build_decoder(...)`

This is **stand-alone** and purely 2D.

1. Input latent `z` `(latent_dim,)`
2. Dense → reshape into a small feature map `(start_side, start_side, start_ch)`
3. While current spatial size `< output_side`:
- Upsample ×2 (bilinear)
- Two Conv2D layers (GELU)
- Reduce channels gradually (`ch = max(ch // 2, 32)`)
4. Output 1-channel image with `Conv2D(1,1)`; forced to `dtype="float32"`.

So the decoder learns: `z -> reconstructed image`.

---

## Notes on design choices

- **Pyramid as depth + Conv3D**: lets the network integrate information across scales (fine-to-coarse) using 3D convolutions, but still keep the depth dimension small and fixed (9).
- **Pooling only spatially**: preserves the 9-scale structure throughout the encoder stages.
- **REE loss**: makes reconstruction error relative to signal energy, which is often more robust when sample magnitudes vary.

---

</details>

In [ ]:
config_solution_autoencoder = {
    'modality': 'u256',
    'data_file': 'data/05_u256.npy',
    'splits': 'data/05_splits.npz',
    'latent': 32,
    'to_level': '1x1',
    'batch': 16,
    'epochs': 10000,
    'lr': 3e-4,
    'c0': 2,
    'cmax': 128,
    'patience': 50,
    'depth_mixer': False,
    'ch_mult': 4,
    'no_mixed_precision': False,
    'pred_batch': 8,
    'predict_device': 'gpu'
}

### Train the autoencoder for all modalities

In [ ]:
for modality in ['u256', 'u128', 'u64', 'u32', 'u16']:  # this may take some time...
    config_solution_autoencoder['modality'] = modality
    config_solution_autoencoder['data_file'] = f'data/05_{modality}.npy'
    solution_autoencoder(**config_solution_autoencoder)

# 5. Train an autoencoder for the stream function

Trains an autoencoder for low-dimensional streamfunction coefficient vectors. It standardizes coefficients using the training split, learns a compact latent representation with whitening regularization (mean, variance, and covariance penalties), saves the best encoder and decoder models, and exports latent vectors for all samples aligned with the original dataset.

<details>

<summary>Options</summary>

- `--coeff_file PATH`
Path to .npy with coeff data (default: `data/05_streamfunction_coeffs.npy`).

- `--splits PATH`
Train/val/test split file (`.npz` or `.json`, default: `data/05_splits.npz`).

- `--latent INT`
Latent dimension of the coefficient representation (default: `32`).

- `--hidden INT`
Width of hidden MLP layers (default: `28`).

- `--batch INT`
Training batch size (default: `128`).

- `--epochs INT`
Maximum number of training epochs (default: `10000`).

- `--patience INT`
Early stopping patience on validation loss (default: `50`).

- `--lr FLOAT`
Learning rate for Adam optimizer (default: `3e-4`).

**Latent whitening penalties**
- `--lam-mu FLOAT`
Penalty weight enforcing zero-mean latent dimensions (default: `1e-3`).
- `--lam-var FLOAT`
Penalty weight enforcing unit variance in latent dimensions (default: `1e-3`).
- `--lam-cov FLOAT`
Penalty weight enforcing near-zero off-diagonal latent covariance (default: `1e-3`).

**Export**
- `--pred-batch INT`
Batch size used when exporting latents (default: `256`).

- `--no-layernorm`
Disable Layer Normalization in hidden layers.

</details>

In [ ]:
config_streamfunction_autoencoder = {
    'coeff_file': 'data/05_streamfunction_coeffs.npy',
    'splits': 'data/05_splits.npz',
    'latent': 32,
    'hidden': 28,
    'batch': 128,
    'epochs': 10000,
    'patience': 50,
    'lr': 3e-4,
    'lam_mu': 1e-3,
    'lam_var': 1e-3,
    'lam_cov': 1e-3,
    'pred_batch': 256,
    'no_layernorm': False
}

In [ ]:
streamfunction_autoencoder(**config_streamfunction_autoencoder)

# 6. Align representations into a shared latent space

This script jointly aligns latent representations from multiple modalities into a shared latent space. Each modality-specific encoder maps inputs to unit-sphere vectors; a joint latent is formed as the normalized mean across modalities. Training progressively shifts reconstruction from self-latents to the joint mean, enforcing alignment while preserving reconstruction quality, and exports aligned encoders, decoders, and latent datasets.

<details>

<summary>Options</summary>

- `--ad INT`
Dimension of the shared latent space (default: `32`).

- `--nonlinear {all,none,coeff}`
Enable nonlinear encoder/decoder heads for all modalities, none, or coefficients only (default: `none`).

- `--align-weight FLOAT`
Weight of the latent alignment loss term (default: `1.0`).

- `--epochs INT`
Maximum number of training epochs (default: `10000`).

- `--batch-size INT`
Batch size for training (default: `64`).

- `--hidden INT`
Hidden width of nonlinear MLP blocks (default: `64`).

- `--activation STR`
Activation function for nonlinear layers (default: `gelu`).

- `--lr FLOAT`
Initial learning rate (default: `1e-2`).

- `--seed INT`
Random seed for reproducibility (default: `42`).

- `--splits PATH`
Train/validation/test split file (default: `data/05_splits.npz`).

- `--joint-anneal-epochs INT`
Number of epochs to ramp the joint-latent weight λ from 0 to 1 (default: `10`; set `0` to start fully joint).

- `--coeff-hidden INT`
Hidden width for coefficient encoders/decoders when nonlinear (default: `128`).

- `--coeff-depth INT`
Depth (number of hidden layers) for coefficient encoders/decoders when nonlinear (default: `2`).

</details>

<details>

<summary>Detailed description of script</summary>

## What this script is doing (big picture)

This is a **“second-level” latent-space alignment and training** script. You already have *per‑modality latent vectors* (each sample has a 32‑D latent for each modality like `u16`, `u32`, …, `coeff`). This script learns:

- **One encoder per modality**: maps that modality’s 32‑D latent into a new shared latent space of dimension `ad` (argument/“aligned dim”).
- **One decoder per modality**: maps **from the shared latent** back to that modality’s original 32‑D latent.

The key design is **strict shared-latent decoding**: during training the decoders are increasingly forced to reconstruct from the **per-sample mean latent** across modalities, not from each modality’s own latent.

Outputs mentioned in the header:
- `models/enc_2_<mod>_ld<ad>.keras` — modality encoder into aligned space
- `models/dec_2_<mod>_ld<ad>.keras` — modality decoder from aligned space
- `latents/lat_2_<mod>_ld<ad>.npy` — aligned unit-sphere latents per modality
- `latents/lat_3_ld<ad>.npy` — joint latent per item (mean over modalities, unit normalized)

---

## 1) Compatibility helpers (Keras 3 vs tf.keras)

The script tries to work in both environments:

- It imports `register_keras_serializable` from:
- `keras.saving` (Keras 3), otherwise
- `tensorflow.keras.utils` (older `tf.keras`)

Then it wraps that in:

```python
def register_serializable(*args, **kwargs):
    return _register_ks(*args, **kwargs)
```


And provides:

```python
def export_model(model, path: str):
    ...
    try:
        model.export(path)     # Keras 3
    except Exception:
        model.save(path)       # tf.keras 2.x SavedModel dir
```


So exporting won’t break depending on your Keras/TensorFlow version.

---

## 2) Constants and file conventions

```python
MODALITY_FILES = {...}
LATENTS_DIR = "latents"
MODELS_DIR = "models"
SPLITS_PATH_DEFAULT = os.path.join("data", "05_splits.npz")
EPS = 1e-8
```


- `MODALITY_FILES` tells the script where to find the **input latents** for each modality.
    - All those files are expected to be shaped **(N, 32)** and be **index-aligned** (row *i* corresponds to the same sample across modalities).

---

## 3) Small utility functions

### Directory creation

```python
def ensure_dirs(*paths):
    for p in paths:
        os.makedirs(p, exist_ok=True)
```


### L2 normalization on tensors

```python
def l2_normalize_tensor(x, axis=-1, eps=EPS):
    return x / (tf.norm(x, axis=axis, keepdims=True) + eps)
```


This enforces that latents live on a **unit hypersphere** (norm ≈ 1). The `eps` avoids division by zero.

### REE loss (Relative Energy Error)

```python
def ree(y_true, y_pred, eps=EPS):
    num = tf.reduce_sum(tf.square(y_true - y_pred), axis=-1)
    den = tf.reduce_sum(tf.square(y_true), axis=-1) + eps
    return num / den
```


REE is basically a per-sample **relative squared error**:
\[
\mathrm{REE} = \frac{\|y - \hat y\|^2}{\|y\|^2 + \epsilon}
\]
It’s scale-invariant: large-magnitude vectors don’t automatically dominate.

---

## 4) Loading the latent data and splits

### `load_modalities(lat_dir)`

- Loads each `.npy` file from `MODALITY_FILES`.
- Verifies:
- it exists
- it is 2D
- second dimension is 32
- Checks all modalities share the same `N`.

Result: a `dict[str, np.ndarray]` like `{ "u16": (N,32), "u32": (N,32), ... }`.

### `load_splits(path, N)`

- If split file doesn’t exist: creates a simple 60/20/20 split by index.
- Else loads `.npz` and tries keys:
- `train`, `val`, `test` or
- `train_idx`, `val_idx`, `test_idx`

### `compute_stats(X_by_mod, train_idx)`

Computes per-modality **mean and std** over training data only, used for standardization:

- `mu = mean(X_train)`
- `sd = std(X_train)` (clipped so tiny std becomes 1.0 to avoid blow-ups)

These stats are then baked into custom layers (see next section) so inference uses the same normalization.

---

## 5) Custom layers (made serialization-safe)

These are implemented as `keras.layers.Layer` subclasses and registered as serializable, so saving/loading models works cleanly without fragile `Lambda` layers.

### `Standardize`
Per-feature normalization: `(x - mean) / std`

- Stores `mean_vec` and `std_vec`
- Converts them to `tf.constant` for use in the graph
- Provides `get_config()` so the layer can be reconstructed from saved config

### `DeStandardize`
Inverse transform: `x * std + mean`

Used at the decoder output to bring standardized values back to original scale.

### `L2Normalize`
Wraps the earlier `l2_normalize_tensor` as a layer.

### `Blend`
A neat trick: a **trainable scalar** mixes two candidate representations:

\[
\text{Blend}(a, b) = \sigma(\alpha)\,a + (1-\sigma(\alpha))\,b
\]

- `alpha_logit` is trained
- `sigmoid(alpha_logit)` ensures the mixing weight is always in (0,1)

This is used to smoothly combine a linear path and a nonlinear path in the encoder.

### `mlp(...)`
A helper to stack `depth` Dense layers of width `hidden` using some activation (e.g., GELU).

---

## 6) Encoder architecture (`build_encoder`)

```python
x_raw = Input((32,))
x = Standardize(...)(x_raw)
z_lin = Dense(ad, activation=None)(x)

if nonlinear:
    h1 = mlp(z_lin, ...)
    z_nl = Dense(ad, activation=None)(h1)
    z = Blend()([z_lin, z_nl])
else:
    z = z_lin

z = L2Normalize()(z)
```


What this means:

- Input is a **32-D “RAW32” latent** from level-1 models.
- It gets standardized (per-feature).
- It gets projected to `ad` dimensions.
- Optionally, a nonlinear refinement path is computed and blended with the linear projection.
- Finally, it’s **L2-normalized**, so every output latent lies on the unit sphere in \(\mathbb{R}^{ad}\).

So each modality encoder outputs a unit vector \(z_k\).

---

## 7) Decoder architecture (`build_decoder`)

Your snippet cuts off mid-function, but the intended decoder (as shown in your open file context) is:

- Input: `z` in $\mathbb{R}^{ad}$
- Output: 32-D vector in the original latent scale

Two cases:

- **Nonlinear decoder**: MLP to 32 plus a **skip/linear** path from input to output, then sum.
- **Linear decoder**: one Dense to 32.

Then `DeStandardize` returns to the raw latent scale.

So each modality decoder learns $(D_k(\cdot)$: $\mathbb{R}^{ad} \to \mathbb{R}^{32}$).

---

## 8) The “shared latent” training logic (the core idea)

The header describes the training objective:

- For each sample, each modality encoder produces a unit latent \(z_k\).
- A **joint** latent is defined as:
\[
$\bar z = \mathrm{L2norm}\left(\frac{1}{K}\sum_k z_k\right)$
\]
- Decoders are trained to reconstruct both:
- from each modality’s own code ($z_k$) (self),
- and from the joint code ($bar z$) (mean).

Loss terms (per the docstring):

- **Self reconstruction**
\[
    $L_\text{self} = \frac{1}{K}\sum_k \mathrm{REE}\big(x_k,\ D_k(z_k)\big)$
\]
- **Mean reconstruction** (the *target regime*)
\[
    $L_\text{mean} = \frac{1}{K}\sum_k \mathrm{REE}\big(x_k,\ D_k(\bar z)\big)$
\]
- **Alignment tightness**
\[
    $L_\text{align} = \frac{1}{K}\sum_k \mathrm{REE}\big(z_k,\ \bar z\big)$
\]

And a curriculum/annealing parameter \(\lambda\) ramps from 0 → 1:

\[
    $L_\text{total} = (1-\lambda)L_\text{self} + \lambda L_\text{mean} + w_\text{align}L_\text{align}$
\]

Interpretation:
- Early training: ($\lambda \approx 0$) → decoders learn to reconstruct from their own modality code (easier).
- Later training: ($\lambda \approx 1$) → decoders are forced to reconstruct from the shared mean latent (the goal).
- Alignment loss prevents each modality code from drifting away from the mean.

---

## 9) Why the unit-sphere constraint matters here

Keeping all ($z_k$) and ($\bar z$) on the unit sphere:

- makes “averaging” directions meaningful (it becomes a kind of consensus direction),
- avoids degenerate solutions where norms blow up to reduce error,
- makes cosine/dot-product similarity behave nicely if you later analyze geometry.

---


</details>

In [ ]:
config_align_latents = {
    'ad': 32,
    'nonlinear': 'none',
    'align_weight': 1.0,
    'epochs': 10000,
    'batch_size': 64,
    'hidden': 64,
    'activation': 'gelu',
    'lr': 1e-2,
    'seed': 42,
    'splits': 'data/05_splits.npz',
    'joint_anneal_epochs': 10,
    'coeff_hidden': 128,
    'coeff_depth': 2
}

In [ ]:
align_latents(**config_align_latents)

# 7. Analyze the alignments

This script evaluates how well second-level modality latents are aligned. It computes pairwise relative energy error (REE) matrices between modalities under raw, centered, and Procrustes-aligned variants, measures each modality’s deviation from the mean latent, optionally applies Procrustes corrections, and constructs a canonical mean latent representation.

<details>

<summary>Options</summary>

- `--mods STR...`
Modalities to include (default: `u16 u32 u64 u128 u256 coeff`).

- `--ld INT`
Latent dimensionality (default: `32`).

- `--latents-dir PATH`
Directory containing `lat_2_<mod>_ld<ld>.npy` files (default: `latents`).

- `--results-dir PATH`
Output directory for CSV result files (default: `results`).

- `--splits PATH`
NPZ file with `train/val/test` indices; if present, Procrustes alignment is fitted on the training split (default: `data/05_splits.npz`).

- `--save-procrustes-corrected`
Save Procrustes-corrected latent arrays as `lat_2p_<mod>_ld<ld>.npy`.

- `--mean-out PATH`
Path to save the canonical mean latent (default: `None`).

</details>

<details>

<summary>Detailed description of script</summary>

## What this script is for (big picture)

This script compares *second-level latent representations* (saved as NumPy arrays) coming from multiple “modalities” (e.g., `u16`, `u32`, …, `coeff`). Each modality provides a matrix:

- `Z_i` with shape **(N, D)**
where `N` = number of samples and `D` = latent dimensionality (`ld`, default 32).

It then computes alignment/consistency metrics between these modality-specific latents using a **relative reconstruction error–style** measure (called `REE` here), under three increasingly “alignment-friendly” conditions:

1. **RAW**: compare latents as-is
2. **CENTERED**: subtract each modality’s column mean before comparing
3. **PROCRUSTES**: after centering, also optimally rotate one latent space to match the other (orthogonal Procrustes)

It also builds a **canonical mean latent** across modalities and measures each modality’s distance to that mean, again under RAW / CENTERED / PROC-to-mean settings.

---

## How to read the printed matrices

- **Lower values** mean closer alignment.
- If **RAW is high** but **CENTERED drops a lot**, the main mismatch is mean/offset.
- If **CENTERED is still high** but **PROCRUSTES drops**, mismatch is largely rotational (spaces are the same up to an orthogonal transform).
- If **PROCRUSTES is still high**, then the latent structures differ more fundamentally (not just offset/rotation).

---


</details>

In [ ]:
config_analyze_alignment = {
    'mods': ('u16', 'u32', 'u64', 'u128', 'u256', 'coeff'),
    'ld': 32,
    'latents_dir': 'latents',
    'results_dir': 'results',
    'splits': 'data/05_splits.npz',
    'save_procrustes_corrected': False,
    'mean_out': None
}

In [ ]:
analyze_alignment(**config_analyze_alignment)

# 8. Fine-tune the chained encoder pipeline

This script fine-tunes a chained encoder pipeline so raw modality inputs map directly to the joint latent representation. It stacks a first-level encoder and an aligned second-level encoder, trains them end-to-end using relative Euclidean error against the canonical latent (`lat_3`), supports freezing/unfreezing stages, and exports the final aligned encoder.

<details>

<summary>Options</summary>

- `--mod {u16,u32,u64,u128,u256,coeff}`
Modality to fine-tune.

- `--lat3 PATH`
Path to the canonical joint latent file (default: `latents/lat_3_ld32.npy`).

- `--epochs INT`
Maximum training epochs (default: `10000`).

- `--batch INT`
Training batch size (default: `128`).

- `--lr FLOAT`
Learning rate for Adam optimizer (default: `1e-3`).

- `--patience INT`
Early stopping patience on validation REE (default: `50`).

- `--splits PATH`
Train/val/test split file (default: `data/05_splits.npz`).

- `--freeze-enc1`
Start training with the first-level encoder frozen.

- `--unfreeze-epoch INT`
If `--freeze-enc1` is set, unfreeze the first encoder at this epoch (default: `0`, never).

</details>

In [ ]:
config_finetune_enc_to_lat = {
    'mod': 'u256',
    'data_prefix': 'data/05_',
    'lat3': 'latents/lat_3_ld32.npy',
    'epochs': 10000,
    'batch': 128,
    'lr': 1e-3,
    'patience': 50,
    'splits': 'data/05_splits.npz',
    'freeze_enc1': False,
    'unfreeze_epoch': 0
}

In [ ]:
for modality in ['u256', 'u128', 'u64', 'u32', 'u16', 'coeff']:
    config_finetune_enc_to_lat['mod'] = modality
    finetune_enc_to_lat(**config_finetune_enc_to_lat)

# 9. Finetune the decoder

This script fine-tunes a chained decoder pipeline so the shared latent representation maps back to modality-specific outputs. It composes an aligned second-level decoder with a first-level decoder, trains them end-to-end using relative Euclidean error in output space, and exports a final decoder capable of reconstructing images or coefficients from the joint latent.

<details>

<summary>Options</summary>

- `--mod {u16,u32,u64,u128,u256,coeff}`
Modality whose decoder is fine-tuned.

- `--lat3 PATH`
Path to aligned joint latents `(N,32)` (default: `latents/lat_3_ld32.npy`).

- `--epochs INT`
Maximum training epochs (default: `10000`).

- `--batch INT`
Training batch size (default: `128`).

- `--lr FLOAT`
Learning rate for Adam optimizer (default: `1e-3`).

- `--patience INT`
Early stopping patience on validation REE (default: `50`).

- `--splits PATH`
Train/validation/test split file (default: `data/05_splits.npz`).

- `--suffix STR`
Optional string appended to output filenames (default: empty).

</details>

In [ ]:
config_finetune_dec_from_lat = {
    'mod': 'u256',
    'lat3': 'latents/lat_3_ld32.npy',
    'epochs': 10000,
    'batch': 128,
    'lr': 1e-3,
    'patience': 50,
    'splits': 'data/05_splits.npz',
    'suffix': ''
}

In [ ]:
for modality in ['u128', 'u64', 'u32', 'u16', 'coeff']:
    config_finetune_dec_from_lat['mod'] = modality
    finetune_dec_from_lat(**config_finetune_dec_from_lat)

# 10. Plot modalities



This script visualizes cross-modal consistency of the aligned latent space. For each sample, it randomly selects one source modality, encodes it with the final encoder, and decodes the shared latent into all modalities. Originals and reconstructions are shown in a 2×6 grid, enabling qualitative inspection of alignment quality.

<details>

<summary>Options</summary>

- `--mods`
Modalities to include (default: `u16 u32 u64 u128`).

- `--split {train,val,test}`
Dataset split to visualize.

- `--seed INT`
Random seed controlling sample and source-modality selection (default: `7`).

- `--num_plots INT`
Number of plots to show (default: 5).

- `--save`
Save figures instead of showing them interactively.

- `--outdir PATH`
Directory for saved figures (default: `results`).

- `--dpi INT`
Resolution of saved PNG images (default: `150`).

- `--cmap STR`
Colormap for solution fields (default: `viridis`).

- `--quiver-step INT`
Subsampling step for velocity quiver plots from coefficients (default: `12`).

- `--source-color STR`
Title color for the source modality (default: `crimson`).

</details>

In [ ]:
config_plot_modalities = {
    'mods': ('u16', 'u32', 'u64', 'u128', 'coeff'),
    'data_prefix': 'data/05_',
    'split': 'val',
    'seed': 7,
    'num_plots': 5,
    'save': False,
    'outdir': 'results',
    'dpi': 150,
    'cmap': 'viridis',
    'quiver_step': 12,
    'source_color': 'crimson'
}

In [ ]:
plot_modalities(**config_plot_modalities)

# 11. Compute errors

This script evaluates quantitative reconstruction quality of the final aligned encoders and decoders. For each modality and data split, it computes relative squared error (REE) for encoding (raw input → joint latent) and decoding (joint latent → reconstructed output), aggregating results into clear per-modality, per-split error tables.

<details>

<summary>Options</summary>

- `--ad INT`
Dimensionality of the aligned joint latent space (default: `32`).

- `--batch INT`
Batch size used when evaluating encoders and decoders (default: `8`).

- `--seed INT`
Random seed for reproducibility (default: `7`).

</details>

In [ ]:
config_compute_errors = {
    'mods': ('u16', 'u32', 'u64', 'u128'),
    'ad': 32,
    'batch': 8,
    'seed': 7
}

In [ ]:
%%capture
enc_tab, dec_tab = errors(**config_compute_errors)

In [ ]:
display(Markdown('### ENCODING REE (enc_3_mod(x_mod) vs latent z)'))
enc_tab

In [ ]:
display(Markdown('### DECODING REE (dec_3_mod(z) vs modality)'))
dec_tab